# Deploy Reusable Fabric Data Agent Hackathon

This organizer notebook provisions the starting environment for a three-hour, hands-on Microsoft Fabric Data Agent hackathon.

The default run creates or reuses a Lakehouse, loads the managed Delta tables, and deploys the Optimized Direct Lake semantic model. The model has sound relationships, explicit measures, business-friendly names, and descriptions, but intentionally starts without synonyms, Prep for AI configuration, AI instructions, Verified Answers, or a Data Agent.

Participants create Data Agents, test questions, diagnose failures, and improve the experience themselves. Fabric IQ Ontology, Data Agent SDK deployment, and organizer-only Prep for AI automation remain optional and disabled by default.

In [ ]:
# Organizer parameters. Keep AI and agent automation disabled for participant-ready deployments.
WORKSPACE_ID = ""
DOMAIN_PROFILE = "water-utilities"
CUSTOM_PROFILE_URL = ""
ASSET_BASE_URL = ""
REPOSITORY_OWNER = "hSushmithaShetty13"
REPOSITORY_NAME = "Water-Utilities"
REPOSITORY_REF = "main"
OVERWRITE_TABLES = True
DEPLOY_OPTIMIZED_MODEL = True
ENABLE_PREP_FOR_AI = False
REFRESH_SEMANTIC_MODELS = True
ENABLE_ONTOLOGY = False
CONFIRM_PREVIEW_DEPLOYMENTS = False
ENABLE_DATA_AGENT = False
PUBLISH_DATA_AGENT = False
LRO_TIMEOUT_SECONDS = 1800
ITEM_DISCOVERY_TIMEOUT_SECONDS = 180
SEMANTIC_REFRESH_TIMEOUT_SECONDS = 900
LAKEHOUSE_SYNC_TIMEOUT_SECONDS = 300

## 1. Load and validate the domain profile

The notebook downloads the profile and reusable definition generator from the selected repository reference. For reproducible events, replace `REPOSITORY_REF` with a release tag or commit SHA.

In [ ]:
import csv
import io
import json
import subprocess
import sys
import time
import types
from urllib.parse import quote
from uuid import UUID

import notebookutils
import pandas as pd
import requests
import sempy.fabric as fabric

context = notebookutils.runtime.context
current_workspace_id = str(
    context.get("currentWorkspaceId") or fabric.get_notebook_workspace_id()
)
target_workspace_id = str(WORKSPACE_ID).strip() or current_workspace_id
UUID(target_workspace_id)

raw_base_url = (
    f"https://raw.githubusercontent.com/{REPOSITORY_OWNER}/"
    f"{REPOSITORY_NAME}/{REPOSITORY_REF}"
)
helper_url = f"{raw_base_url}/deployment/hackathon_deployer.py"
profile_url = CUSTOM_PROFILE_URL or (
    f"{raw_base_url}/config/domains/{quote(DOMAIN_PROFILE)}.json"
)
asset_base_url = ASSET_BASE_URL.rstrip("/") or raw_base_url

helper_response = requests.get(helper_url, timeout=180)
helper_response.raise_for_status()
deployer = types.ModuleType("hackathon_deployer")
exec(compile(helper_response.text, helper_url, "exec"), deployer.__dict__)

profile_response = requests.get(profile_url, timeout=180)
profile_response.raise_for_status()
profile = profile_response.json()
deployer.validate_profile(profile)

deployment_results = []
semantic_model_ids = {}
print("Domain:", profile["domain"]["displayName"])
print("Current workspace:", current_workspace_id)
print("Target workspace:", target_workspace_id)
print("Profile:", profile_url)

## 2. Resolve the workspace folder and Lakehouse

All Fabric control-plane calls are authenticated with the notebook identity. Existing artifacts are reused by exact display name.

In [ ]:
FABRIC_API = "https://api.fabric.microsoft.com/v1"
base_headers = {
    "Content-Type": "application/json",
    "x-ms-fabric-skill": "reusable-data-agent-hackathon",
}


def authenticated_headers(headers=None):
    return {
        **base_headers,
        **(headers or {}),
        "Authorization": f"Bearer {notebookutils.credentials.getToken('pbi')}",
    }


def send_request(method, url, *, body=None, headers=None, timeout=180):
    for attempt in range(2):
        response = requests.request(
            method,
            url,
            headers=authenticated_headers(headers),
            json=body,
            timeout=timeout,
        )
        if response.status_code != 401 or attempt == 1:
            return response
    return response


def fabric_request(method, url, *, body=None, headers=None, expect_result=False):
    response = send_request(method, url, body=body, headers=headers)
    if response.status_code not in (200, 201, 202):
        raise RuntimeError(
            f"{method} {url} failed: HTTP {response.status_code} {response.text}"
        )
    if response.status_code != 202:
        return response.json() if response.content else None

    operation_id = response.headers.get("x-ms-operation-id") or response.headers.get(
        "Operation-Id"
    )
    if not operation_id:
        raise RuntimeError(f"{method} {url} returned 202 without an operation ID")
    operation_url = f"{FABRIC_API}/operations/{operation_id}"
    deadline = time.monotonic() + LRO_TIMEOUT_SECONDS
    delay = 5.0
    while time.monotonic() < deadline:
        operation = send_request("GET", operation_url, headers=headers, timeout=60)
        operation.raise_for_status()
        operation_payload = operation.json()
        status = operation_payload.get("status")
        if status == "Succeeded":
            if expect_result:
                result = send_request(
                    "GET",
                    f"{operation_url}/result",
                    headers=headers,
                    timeout=180,
                )
                result.raise_for_status()
                return result.json()
            return operation_payload
        if status in {"Failed", "Cancelled"}:
            raise RuntimeError(
                f"Fabric operation {operation_id} {status}: "
                f"{json.dumps(operation_payload.get('error', {}))}"
            )
        retry_after = operation.headers.get("Retry-After")
        time.sleep(float(retry_after) if retry_after else delay)
        delay = min(delay * 1.5, 30.0)
    raise TimeoutError(
        f"Fabric operation {operation_id} did not complete in "
        f"{LRO_TIMEOUT_SECONDS} seconds"
    )


def list_fabric_values(url, headers=None):
    values = []
    next_url = url
    while next_url:
        payload = fabric_request("GET", next_url, headers=headers) or {}
        values.extend(payload.get("value", []))
        next_url = payload.get("continuationUri")
    return values


def find_item(display_name, item_type):
    items = list_fabric_values(
        f"{FABRIC_API}/workspaces/{target_workspace_id}/items?type={item_type}"
    )
    return next((item for item in items if item.get("displayName") == display_name), None)


def wait_for_item(display_name, item_type):
    deadline = time.monotonic() + ITEM_DISCOVERY_TIMEOUT_SECONDS
    delay = 2.0
    while time.monotonic() < deadline:
        item = find_item(display_name, item_type)
        if item is not None:
            return item
        time.sleep(delay)
        delay = min(delay * 1.5, 15.0)
    raise TimeoutError(
        f"{item_type} {display_name!r} was not discoverable after "
        f"{ITEM_DISCOVERY_TIMEOUT_SECONDS} seconds"
    )


def ensure_folder(display_name):
    url = f"{FABRIC_API}/workspaces/{target_workspace_id}/folders"
    folders = list_fabric_values(url)
    existing = next(
        (
            folder
            for folder in folders
            if folder.get("displayName") == display_name
            and not folder.get("parentFolderId")
        ),
        None,
    )
    if existing:
        return str(existing["id"])
    created = fabric_request("POST", url, body={"displayName": display_name})
    return str(created["id"])


def ensure_item_in_folder(item_id, folder_id):
    item_url = f"{FABRIC_API}/workspaces/{target_workspace_id}/items/{item_id}"
    item = fabric_request("GET", item_url)
    if str(item.get("folderId", "")) == str(folder_id):
        return False
    fabric_request("POST", f"{item_url}/move", body={"targetFolderId": folder_id})
    return True


def ensure_lakehouse(display_name, folder_id):
    existing = find_item(display_name, "Lakehouse")
    if existing:
        ensure_item_in_folder(existing["id"], folder_id)
        return str(existing["id"]), "Reused"
    body = {"displayName": display_name, "type": "Lakehouse", "folderId": folder_id}
    fabric_request(
        "POST",
        f"{FABRIC_API}/workspaces/{target_workspace_id}/items",
        body=body,
    )
    created = wait_for_item(display_name, "Lakehouse")
    return str(created["id"]), "Created"


folder_id = ensure_folder(profile["artifacts"]["folder"])
lakehouse_id, lakehouse_action = ensure_lakehouse(
    profile["artifacts"]["lakehouse"], folder_id
)
deployment_results.append(
    {
        "type": "Lakehouse",
        "name": profile["artifacts"]["lakehouse"],
        "action": lakehouse_action,
        "id": lakehouse_id,
    }
)
print("Workspace folder ID:", folder_id)
print("Lakehouse ID:", lakehouse_id)

## 3. Load profile data into managed Delta tables

Each source file is validated against the profile before writing. The notebook writes to the target Lakehouse by immutable workspace and Lakehouse IDs, then waits for the Lakehouse's own table metadata (the layer backing the SQL analytics endpoint and Direct Lake) to catch up with the freshly written tables before the semantic model is deployed.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, StructField, StructType

endpoint_value = notebookutils.conf.get("trident.onelake.endpoint")
if not endpoint_value:
    raise RuntimeError("The Fabric runtime did not provide the OneLake endpoint")
onelake_endpoint = endpoint_value.removeprefix("https://").rstrip("/")
if not target_workspace_id or not lakehouse_id:
    raise RuntimeError("Workspace and Lakehouse IDs must be resolved before loading data")
table_root = (
    f"abfss://{target_workspace_id}@{onelake_endpoint}/"
    f"{lakehouse_id}/Tables"
)


def source_url(relative_path):
    return f"{asset_base_url}/{quote(relative_path, safe='/')}"


def cast_profile_column(frame, column):
    name = column["source"]
    data_type = column["type"]
    if data_type == "int64":
        return frame.withColumn(name, F.col(name).cast("long"))
    if data_type == "double":
        return frame.withColumn(name, F.col(name).cast("double"))
    if data_type == "boolean":
        return frame.withColumn(
            name,
            F.when(F.lower(F.col(name)).isin("true", "1", "yes"), F.lit(True))
            .when(F.lower(F.col(name)).isin("false", "0", "no"), F.lit(False))
            .otherwise(F.lit(None).cast("boolean")),
        )
    if data_type in {"date", "dateTime"}:
        source_format = "yyyy-MM-dd" if data_type == "date" else "yyyy-MM-dd HH:mm:ss"
        parsed = F.to_date(F.col(name), source_format) if data_type == "date" else F.to_timestamp(F.col(name), source_format)
        invalid_value = (
            frame.where(F.col(name).isNotNull() & parsed.isNull())
            .select(name)
            .limit(1)
            .collect()
        )
        if invalid_value:
            raise ValueError(
                f"Could not parse {data_type} value {invalid_value[0][name]!r} "
                f"in {name!r}; expected source format {source_format!r}."
            )
        return frame.withColumn(name, parsed)
    return frame


write_mode = "overwrite" if OVERWRITE_TABLES else "errorifexists"
for table in profile["tables"]:
    response = requests.get(source_url(table["sourcePath"]), timeout=180)
    response.raise_for_status()
    reader = csv.DictReader(io.StringIO(response.text))
    expected_columns = [column["source"] for column in table["columns"]]
    if reader.fieldnames != expected_columns:
        raise ValueError(
            f"Header mismatch for {table['sourcePath']}: "
            f"expected {expected_columns}, found {reader.fieldnames}"
        )
    rows = list(reader)
    schema = StructType([StructField(name, StringType(), True) for name in expected_columns])
    values = [tuple(row.get(name) or None for name in expected_columns) for row in rows]
    frame = spark.createDataFrame(values, schema)
    for column in table["columns"]:
        frame = cast_profile_column(frame, column)
    destination = f"{table_root}/{table['lakehouseName']}"
    (
        frame.write.format("delta")
        .mode(write_mode)
        .option("overwriteSchema", "true")
        .save(destination)
    )
    deployment_results.append(
        {
            "type": "Lakehouse table",
            "name": table["lakehouseName"],
            "action": "Loaded",
            "rows": len(rows),
        }
    )
    print(f"Loaded {table['lakehouseName']}: {len(rows)} rows")


def list_lakehouse_table_names():
    names = set()
    url = f"{FABRIC_API}/workspaces/{target_workspace_id}/lakehouses/{lakehouse_id}/tables"
    while url:
        payload = fabric_request("GET", url) or {}
        names.update(
            table.get("name") for table in (payload.get("data") or payload.get("value") or [])
        )
        continuation = payload.get("continuationToken")
        url = (
            f"{FABRIC_API}/workspaces/{target_workspace_id}/lakehouses/{lakehouse_id}/tables"
            f"?continuationToken={continuation}"
            if continuation
            else None
        )
    return names


def wait_for_lakehouse_table_sync(expected_names):
    # Writing Delta files straight to the Tables/ path skips the catalog, so the SQL
    # analytics endpoint (and therefore Direct Lake) can lag behind the physical files.
    deadline = time.monotonic() + LAKEHOUSE_SYNC_TIMEOUT_SECONDS
    delay = 5.0
    missing = set(expected_names)
    while time.monotonic() < deadline:
        missing = set(expected_names) - list_lakehouse_table_names()
        if not missing:
            print("Lakehouse table metadata is in sync; safe to deploy the semantic model.")
            return
        time.sleep(delay)
        delay = min(delay * 1.5, 30.0)
    raise TimeoutError(
        f"Lakehouse metadata sync did not detect these tables in "
        f"{LAKEHOUSE_SYNC_TIMEOUT_SECONDS} seconds: {sorted(missing)}"
    )


wait_for_lakehouse_table_sync([table["lakehouseName"] for table in profile["tables"]])

## 4. Generate and deploy Direct Lake semantic models

The Optimized model is generated as a complete TMDL definition in memory. Existing models receive complete definition updates, preventing partial-definition loss and removing the need for PBIP or PBIX artifacts.

In [ ]:
def get_semantic_definition(item_id):
    return fabric_request(
        "POST",
        (
            f"{FABRIC_API}/workspaces/{target_workspace_id}/semanticModels/"
            f"{item_id}/getDefinition?format=TMDL"
        ),
        body={},
        expect_result=True,
    )


def deploy_semantic_model(model_key, artifact_name, include_copilot=False):
    parts = deployer.render_semantic_model_parts(
        profile,
        model_key,
        target_workspace_id,
        lakehouse_id,
        onelake_endpoint,
    )
    if include_copilot:
        parts.update(deployer.render_copilot_parts(profile))
    existing = find_item(artifact_name, "SemanticModel")
    if existing:
        item_id = str(existing["id"])
        fabric_request(
            "POST",
            (
                f"{FABRIC_API}/workspaces/{target_workspace_id}/semanticModels/"
                f"{item_id}/updateDefinition"
            ),
            body=deployer.definition_payload(parts, "TMDL"),
        )
        action = "Updated"
    else:
        fabric_request(
            "POST",
            f"{FABRIC_API}/workspaces/{target_workspace_id}/semanticModels",
            body=deployer.semantic_model_create_payload(artifact_name, parts),
        )
        created = wait_for_item(artifact_name, "SemanticModel")
        item_id = str(created["id"])
        action = "Created"
    ensure_item_in_folder(item_id, folder_id)
    semantic_model_ids[model_key] = item_id
    deployment_results.append(
        {
            "type": "Semantic model",
            "name": artifact_name,
            "action": action,
            "id": item_id,
        }
    )
    print(f"{action} semantic model: {artifact_name}")
    return item_id


if DEPLOY_OPTIMIZED_MODEL:
    deploy_semantic_model(
        "optimized",
        profile["artifacts"]["optimizedModel"],
        include_copilot=ENABLE_PREP_FOR_AI,
    )

## 5. Optional organizer-only Prep for AI automation

This stage is disabled by default so participants can configure synonyms, AI Data Schema scope, AI instructions, example prompts, and Verified Answers during the hackathon. The generated TMDL already includes deterministic lineage tags for every table, column, and measure.

Set `ENABLE_PREP_FOR_AI=True` only when an organizer needs a fully configured demonstration environment. Verified Answers always remain a manual task because they must reference saved report visuals and be tested in the live authoring experience.

In [ ]:
if ENABLE_PREP_FOR_AI:
    optimized_id = semantic_model_ids.get("optimized")
    if not optimized_id:
        optimized_item = find_item(profile["artifacts"]["optimizedModel"], "SemanticModel")
        if not optimized_item:
            raise RuntimeError("The optimized semantic model is required for Prep for AI")
        optimized_id = str(optimized_item["id"])
        semantic_model_ids["optimized"] = optimized_id

    optimized_parts = deployer.render_semantic_model_parts(
        profile,
        "optimized",
        target_workspace_id,
        lakehouse_id,
        onelake_endpoint,
    )
    optimized_parts.update(deployer.render_copilot_parts(profile))
    optimized_parts["Copilot/schema.json"] = deployer.render_copilot_schema(
        profile, optimized_parts
    )
    fabric_request(
        "POST",
        (
            f"{FABRIC_API}/workspaces/{target_workspace_id}/semanticModels/"
            f"{optimized_id}/updateDefinition"
        ),
        body=deployer.definition_payload(optimized_parts, "TMDL"),
    )
    deployment_results.append(
        {
            "type": "Semantic model AI metadata",
            "name": profile["artifacts"]["optimizedModel"],
            "action": "Updated",
            "id": optimized_id,
        }
    )
    print("Applied AI instructions, schema selection, and example prompts.")
    print("Verified Answer candidates to configure from saved report visuals:")
    for candidate in profile["ai"]["verifiedAnswerCandidates"]:
        print(f"- {candidate['question']} -> [{candidate['measure']}]")
else:
    print("Prep for AI metadata is disabled.")

## 6. Optional Fabric IQ Ontology (preview)

Ontology deployment is disabled by default. Set both `ENABLE_ONTOLOGY` and `CONFIRM_PREVIEW_DEPLOYMENTS` to `True` after reviewing the entity and relationship proposal. Re-running uses deterministic IDs derived from the domain profile.

In [ ]:
if ENABLE_ONTOLOGY:
    if not CONFIRM_PREVIEW_DEPLOYMENTS:
        raise ValueError(
            "Review the proposed ontology and set CONFIRM_PREVIEW_DEPLOYMENTS=True"
        )
    ontology_name = profile["artifacts"]["ontology"]
    ontology_headers = {"x-ms-fabric-skill": "fabriciq-ontology-cli"}
    ontology_parts = deployer.render_ontology_parts(
        profile, target_workspace_id, lakehouse_id
    )
    print("Ontology proposal:")
    for entity in profile.get("ontology", {}).get("entities", []):
        print(f"- Entity: {entity['name']} -> {entity['table']}")
    for relationship in profile.get("ontology", {}).get("relationships", []):
        print(
            f"- Relationship: {relationship['from']} -> "
            f"{relationship['to']} ({relationship['name']})"
        )

    existing_ontology = find_item(ontology_name, "Ontology")
    if existing_ontology:
        ontology_id = str(existing_ontology["id"])
        # Read current state before replacing this notebook-managed definition.
        fabric_request(
            "POST",
            (
                f"{FABRIC_API}/workspaces/{target_workspace_id}/items/"
                f"{ontology_id}/getDefinition"
            ),
            body={},
            headers=ontology_headers,
            expect_result=True,
        )
        fabric_request(
            "POST",
            (
                f"{FABRIC_API}/workspaces/{target_workspace_id}/items/"
                f"{ontology_id}/updateDefinition?updateMetadata=true"
            ),
            body=deployer.definition_payload(ontology_parts),
            headers=ontology_headers,
        )
        ontology_action = "Updated"
    else:
        create_body = {
            "displayName": ontology_name,
            "type": "Ontology",
            "folderId": folder_id,
            **deployer.definition_payload(ontology_parts),
        }
        fabric_request(
            "POST",
            f"{FABRIC_API}/workspaces/{target_workspace_id}/items",
            body=create_body,
            headers=ontology_headers,
        )
        created_ontology = wait_for_item(ontology_name, "Ontology")
        ontology_id = str(created_ontology["id"])
        ontology_action = "Created"
    ensure_item_in_folder(ontology_id, folder_id)
    deployment_results.append(
        {
            "type": "Ontology",
            "name": ontology_name,
            "action": ontology_action,
            "id": ontology_id,
        }
    )
else:
    print("Ontology deployment is disabled.")

## 7. Optional Data Agent (preview SDK)

The Data Agent stage uses `fabric-data-agent-sdk==0.1.28a0`, configures staging sources by profile, and publishes only when `PUBLISH_DATA_AGENT=True`. It is disabled by default so the stable Lakehouse and semantic-model deployment does not depend on a preview SDK.

In [ ]:
if ENABLE_DATA_AGENT:
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "fabric-data-agent-sdk==0.1.28a0",
            "mcp==1.23.3",
        ]
    )
    from fabric.dataagent.client import create_data_agent

    agent_name = profile["artifacts"]["dataAgent"]
    data_agent = create_data_agent(agent_name, workspace_id=target_workspace_id)
    leaf_hints = ("column", "measure", "parameter", "returnvalue")

    def element_name(element):
        return element.get("displayName") or element.get("name")

    def is_leaf(element):
        element_type = str(element.get("type", "")).lower()
        return any(hint in element_type for hint in leaf_hints)

    def datasource_name(datasource):
        configuration = datasource.get_configuration(stage="staging")
        return configuration.get("displayName") or configuration.get("display_name")

    def resolve_datasource(name):
        for datasource in data_agent.list_datasources(stage="staging"):
            if datasource_name(datasource) == name:
                return datasource
        return None

    def ensure_datasource(name):
        datasource = resolve_datasource(name)
        if datasource is not None:
            return datasource
        data_agent.add_staging_datasource(
            name,
            workspace_id_or_name=target_workspace_id,
        )
        for _ in range(30):
            time.sleep(3)
            datasource = resolve_datasource(name)
            if datasource is not None:
                return datasource
        raise RuntimeError(f"Datasource {name!r} did not appear in staging")

    def datasource_children(datasource, root_id):
        values = []
        continuation_token = None
        while True:
            response = datasource.get_elements(
                stage="staging",
                root_id=root_id,
                continuation_token=continuation_token,
            )
            values.extend(response.get("value", []))
            continuation_token = response.get("continuationToken")
            if not continuation_token:
                return values

    def walk_elements(datasource, root_id=None):
        for element in datasource_children(datasource, root_id):
            yield element
            if not is_leaf(element):
                yield from walk_elements(datasource, element["id"])

    def select_by_name(datasource, names):
        selected = set()
        for element in walk_elements(datasource):
            name = element_name(element)
            if name in names and not is_leaf(element):
                datasource.update_element(element["id"], is_selected=True)
                selected.add(name)
        missing = set(names) - selected
        if missing:
            raise ValueError(f"Datasource objects not found: {sorted(missing)}")
        return sorted(selected)

    for source in profile["agent"]["sources"]:
        artifact_name = profile["artifacts"][source["artifact"]]
        datasource = ensure_datasource(artifact_name)
        selected = select_by_name(datasource, source["objects"])
        datasource.update_configuration(
            description=source["description"],
            instructions=source["instructions"],
        )
        examples = {
            example["question"]: example["query"]
            for example in source.get("examples", [])
            if example.get("query")
        }
        if examples:
            existing = datasource.get_fewshots(stage="staging")
            question_column = next(
                (column for column in existing.columns if str(column).lower() == "question"),
                None,
            )
            existing_questions = (
                set(existing[question_column].astype(str))
                if question_column is not None
                else set()
            )
            missing_examples = {
                question: query
                for question, query in examples.items()
                if question not in existing_questions
            }
            if missing_examples:
                datasource.add_fewshots(missing_examples)
        print(f"Configured {artifact_name}: {selected}")

    data_agent.update_settings(ai_instructions=profile["agent"]["instructions"])
    if PUBLISH_DATA_AGENT:
        data_agent.publish_staging(description=profile["agent"]["description"])
        agent_action = "Published"
    else:
        agent_action = "Staged"
    deployment_results.append(
        {"type": "Data Agent", "name": agent_name, "action": agent_action}
    )
else:
    print("Data Agent deployment is disabled.")

## 8. Refresh and verify deployment

The final stage refreshes each deployed Direct Lake semantic model after all definition updates, waits for the refresh to complete, confirms every stable-core artifact by exact name, and validates every expected OneLake table path. Preview artifacts are reported only when enabled.

In [ ]:
from delta.tables import DeltaTable

POWER_BI_API = "https://api.powerbi.com/v1.0/myorg"


def refresh_semantic_model(model_id, model_name):
    refresh_url = (
        f"{POWER_BI_API}/groups/{target_workspace_id}/datasets/{model_id}/refreshes"
    )
    response = send_request(
        "POST",
        refresh_url,
        body={"type": "Full", "commitMode": "transactional", "retryCount": 1},
    )
    if response.status_code not in (200, 202):
        raise RuntimeError(
            f"Refreshing {model_name} failed: HTTP {response.status_code} {response.text}"
        )

    polling_url = response.headers.get("Location")
    request_id = response.headers.get("RequestId")
    if not polling_url and request_id:
        polling_url = f"{refresh_url}/{request_id}"
    if not polling_url:
        raise RuntimeError(f"Refreshing {model_name} returned no polling location")

    deadline = time.monotonic() + SEMANTIC_REFRESH_TIMEOUT_SECONDS
    delay = 5.0
    while time.monotonic() < deadline:
        status_response = send_request("GET", polling_url, timeout=60)
        status_response.raise_for_status()
        refresh = status_response.json()
        status = refresh.get("status")
        if status == "Completed":
            print(f"Refreshed semantic model: {model_name}")
            return
        if status in {"Failed", "Cancelled", "Disabled"}:
            raise RuntimeError(
                f"Refreshing {model_name} ended with {status}: "
                f"{json.dumps(refresh.get('messages', []))}"
            )
        time.sleep(delay)
        delay = min(delay * 1.5, 30.0)
    raise TimeoutError(
        f"Refreshing {model_name} did not complete in "
        f"{SEMANTIC_REFRESH_TIMEOUT_SECONDS} seconds"
    )


if REFRESH_SEMANTIC_MODELS:
    models_to_refresh = []
    if DEPLOY_OPTIMIZED_MODEL or ENABLE_PREP_FOR_AI:
        models_to_refresh.append((
            "optimized", profile["artifacts"]["optimizedModel"]
        ))
    for model_key, model_name in models_to_refresh:
        model_id = semantic_model_ids.get(model_key)
        if not model_id:
            model_item = find_item(model_name, "SemanticModel")
            if not model_item:
                raise RuntimeError(f"Cannot refresh missing semantic model: {model_name}")
            model_id = str(model_item["id"])
            semantic_model_ids[model_key] = model_id
        refresh_semantic_model(model_id, model_name)

required_items = [
    (profile["artifacts"]["lakehouse"], "Lakehouse"),
]
if DEPLOY_OPTIMIZED_MODEL or ENABLE_PREP_FOR_AI or ENABLE_DATA_AGENT:
    required_items.append((profile["artifacts"]["optimizedModel"], "SemanticModel"))
if ENABLE_ONTOLOGY:
    required_items.append((profile["artifacts"]["ontology"], "Ontology"))

missing_items = [
    f"{item_type}:{name}"
    for name, item_type in required_items
    if find_item(name, item_type) is None
]
if missing_items:
    raise AssertionError(f"Missing Fabric items: {missing_items}")

missing_tables = []
for table in profile["tables"]:
    path = f"{table_root}/{table['lakehouseName']}"
    if not DeltaTable.isDeltaTable(spark, path):
        missing_tables.append(table["lakehouseName"])
if missing_tables:
    raise AssertionError(f"Missing or invalid Delta tables: {missing_tables}")

summary = pd.DataFrame(deployment_results)
display(summary)
print("Participant-ready deployment verified.")
print("Participant challenge: create Data Agents, run the baseline questions, and improve behavior with synonyms, Prep for AI, and agent instructions.")
print("Verified Answers remain a manual challenge using saved report visuals.")
if ENABLE_ONTOLOGY:
    print("Manual follow-up: refresh the Ontology graph model after upstream data changes.")